# 卷积神经网络（CNN）

两层卷积 + 池化 + 全连接，与仓库 `cnn.py` 结构一致。输入保持 `(N, 1, 28, 28)`。


In [1]:
from pathlib import Path

import torch
import torch.nn as nn
import torch.utils.data as Data
import torchvision

torch.manual_seed(1)


## 超参数


In [2]:
EPOCH = 1
BATCH_SIZE = 50
LR = 0.001

TEST_N = 2000


## 数据与 DataLoader


In [3]:
import numpy as np
from pathlib import Path

import torchvision
from mnist_from_raw import MNISTNumpyDataset, load_all_numpy, raw_files_available

if raw_files_available():
    train_x, train_y, te_imgs, te_lbls = load_all_numpy()
    train_data = MNISTNumpyDataset(train_x, train_y)
    train_loader = Data.DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=True)
    test_x = torch.from_numpy(np.ascontiguousarray(te_imgs[:TEST_N])).unsqueeze(1).float().div_(255.0)
    test_y = torch.from_numpy(te_lbls[:TEST_N].copy()).long()
    print("数据来源: data/raw")
else:
    MNIST_ROOT = Path("./mnist")
    download = not MNIST_ROOT.is_dir() or not any(MNIST_ROOT.iterdir())
    train_data = torchvision.datasets.MNIST(
        root=str(MNIST_ROOT),
        train=True,
        transform=torchvision.transforms.ToTensor(),
        download=download,
    )
    train_loader = Data.DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=True)
    test_data = torchvision.datasets.MNIST(
        root=str(MNIST_ROOT), train=False, download=download
    )
    test_x = torch.unsqueeze(test_data.test_data, dim=1).type(torch.FloatTensor)[:TEST_N] / 255.0
    test_y = test_data.test_labels[:TEST_N]
    print("数据来源: torchvision ->", MNIST_ROOT.resolve())
print("train batches (approx):", len(train_loader))
print("test_x:", test_x.shape, "test_y:", test_y.shape)


数据来源: data/raw
train batches (approx): 1200
test_x: torch.Size([2000, 1, 28, 28]) test_y: torch.Size([2000])


/tmp/ipykernel_3281921/156012612.py:11: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  test_x = torch.from_numpy(np.ascontiguousarray(te_imgs[:TEST_N])).unsqueeze(1).float().div_(255.0)


## 模型定义


In [4]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


model = CNN()
print(model)


CNN(
  (conv1): Sequential(
    (0): Conv2d(1, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Linear(in_features=1568, out_features=10, bias=True)
)


## 优化器与损失


In [5]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()


## 训练


In [6]:
def accuracy(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()

for epoch in range(EPOCH):
    for step, (b_x, b_y) in enumerate(train_loader):
        logits = model(b_x)
        loss = loss_fn(logits, b_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 50 == 0:
            with torch.no_grad():
                acc = accuracy(model(test_x), test_y)
            print(f"epoch={epoch} step={step} loss={loss.item():.4f} test_acc={acc:.4f}")


/data1/zdguo/document-parsing/alextools/experiment/MNIST/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:869: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


epoch=0 step=0 loss=2.3034 test_acc=0.1210
epoch=0 step=50 loss=0.4083 test_acc=0.8265
epoch=0 step=100 loss=0.5549 test_acc=0.8680
epoch=0 step=150 loss=0.2233 test_acc=0.8955
epoch=0 step=200 loss=0.1272 test_acc=0.9345
epoch=0 step=250 loss=0.1658 test_acc=0.9410
epoch=0 step=300 loss=0.0554 test_acc=0.9500
epoch=0 step=350 loss=0.1249 test_acc=0.9550
epoch=0 step=400 loss=0.0346 test_acc=0.9590
epoch=0 step=450 loss=0.1015 test_acc=0.9655
epoch=0 step=500 loss=0.1825 test_acc=0.9670
epoch=0 step=550 loss=0.0934 test_acc=0.9675
epoch=0 step=600 loss=0.0183 test_acc=0.9725
epoch=0 step=650 loss=0.1446 test_acc=0.9705
epoch=0 step=700 loss=0.0966 test_acc=0.9650
epoch=0 step=750 loss=0.2095 test_acc=0.9580
epoch=0 step=800 loss=0.0380 test_acc=0.9640
epoch=0 step=850 loss=0.0505 test_acc=0.9720
epoch=0 step=900 loss=0.0950 test_acc=0.9770
epoch=0 step=950 loss=0.0199 test_acc=0.9765
epoch=0 step=1000 loss=0.1993 test_acc=0.9775
epoch=0 step=1050 loss=0.0306 test_acc=0.9775
epoch=0 ste

## 预测样例


In [7]:
model.eval()
with torch.no_grad():
    pred = model(test_x[:10]).argmax(dim=1).cpu().numpy()
print("pred:", pred)
print("true:", test_y[:10].numpy())


pred: [7 2 1 0 4 1 4 9 5 9]
true: [7 2 1 0 4 1 4 9 5 9]
